# Practical-04: Prompt Engineering Experiments

### Setup the System configuration & check AWS config

In [1]:
import sys, os
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from utils.config import config
print("Region:", config.region_name)
print("Default model:", config.default_model_id)

Region: us-east-1
Default model: amazon.nova-lite-v1:0


## Zero-Shot vs Few-Shot vs Chain-of-Thought
- **Zero-shot** — task instructions + category list only. Cheapest, fastest, no worked examples.
- **Few-shot** — adds examples before the real ticket, to prime the exact decision boundary we want.
- **Chain-of-thought (CoT)** — solves the complex task which is ambiguous to the problem

In [2]:
from src.variants import compare_variants, CATEGORIES

print("Categories:", CATEGORIES)

Categories: ['Payroll', 'Leave & Attendance', 'IT Support', 'Policy & General']


In [3]:
test_tickets = [
    "I haven't received my reimbursement for last month's travel expenses.",   # clear: Payroll
    "How many casual leaves do I have left this quarter?",                     # clear: Leave & Attendance
    "My work laptop keeps disconnecting from the VPN, please help.",           # clear: IT Support
    # Deliberately ambiguous — mentions both payroll AND leave:
    "I took unpaid leave last week but my salary this month looks wrong, can you check both?",
]

for ticket in test_tickets:

    print("TICKET:", ticket)

    results = compare_variants(ticket)
    for variant_name, res in results.items():
        print(f"\n[{variant_name.upper()}]")
        print(res["text"])
    print()

TICKET: I haven't received my reimbursement for last month's travel expenses.

[ZERO_SHOT]
Policy & General

[FEW_SHOT]
Payroll

[COT]
Reasoning: The ticket mentions "reimbursement for last month's travel expenses," which relates to financial transactions and expense management. This topic is most closely associated with payroll processes, as reimbursements for expenses are typically handled through payroll systems.

Category: Payroll

TICKET: How many casual leaves do I have left this quarter?

[ZERO_SHOT]
Leave & Attendance

[FEW_SHOT]
Leave & Attendance

[COT]
Reasoning: The ticket is asking about the number of casual leaves remaining for the quarter, which pertains to tracking and managing leave balances. This is directly related to the employee's leave entitlements and usage, which falls under the category of Leave & Attendance.

Category: Leave & Attendance

TICKET: My work laptop keeps disconnecting from the VPN, please help.

[ZERO_SHOT]
IT Support

[FEW_SHOT]
IT Support

[COT]

## Observations of above techniques

| Technique | Tokens used (roughly) | Handles the ambiguous ticket well? | Notes |
|---|---|---|---|
| Zero-shot | Lowest | Often picks just one topic, sometimes the wrong one | Fastest/cheapest, fine for clear-cut tickets |
| Few-shot | Medium | Better than zero-shot — examples nudge it toward the intended boundary | Good default when categories are easily confused |
| Chain-of-thought | Highest (extra reasoning tokens) | Usually best — reasoning surfaces that both topics were mentioned, then picks the dominant one | Most transparent (you can audit *why*), but hardest to parse programmatically and slowest/costliest |

## System Message/ Persona comparison

In [4]:
# Same user prompt, sent twice — once with the system message set to senior data analyst, once to creative writer (see `src/roles.py` for the exact persona text). Any difference in the output is attributable to the system message alone, since the user prompt never changes.

from src.roles import compare_roles, SYSTEM_PROMPTS

print("--- senior_data_analyst system prompt ---")
print(SYSTEM_PROMPTS["senior_data_analyst"])
print("\n--- creative_writer system prompt ---")
print(SYSTEM_PROMPTS["creative_writer"])


--- senior_data_analyst system prompt ---
You are a senior data analyst. Respond precisely and objectively. Use structured, factual language, quantify things where possible, avoid embellishment, and get to the point quickly.

--- creative_writer system prompt ---
You are an imaginative creative writer. Respond with vivid, expressive language, use metaphor and narrative flair where it fits, and prioritize engaging, colorful phrasing over brevity.


In [5]:
shared_prompt = "Explain why employee turnover is a problem worth solving."

results = compare_roles(shared_prompt)

for role_name, res in results.items():
    print("=" * 90)
    print(f"ROLE: {role_name}")
    print("=" * 90)
    print(res["text"])
    print()

ROLE: senior_data_analyst
**Employee Turnover: A Problem Worth Solving**

1. **Cost Implications**
   - **Recruitment Costs**: Average cost per hire ranges from $4,000 to $5,000.
   - **Training Costs**: Retraining a new employee can cost up to 6-9 months of their salary.
   - **Productivity Loss**: Turnover can lead to a 30-200% loss in productivity for the position.

2. **Operational Disruption**
   - **Skill Gaps**: High turnover can result in skill shortages, impacting project timelines.
   - **Knowledge Loss**: Departing employees take institutional knowledge with them, leading to inefficiencies.

3. **Employee Morale**
   - **Decreased Morale**: High turnover rates can demoralize remaining employees, reducing overall morale.
   - **Increased Workload**: Remaining staff may need to cover for departing colleagues, leading to burnout.

4. **Brand Reputation**
   - **Employer Brand**: High turnover can damage a company’s reputation as an employer of choice.
   - **Customer Perception

### second prompt, to check the pattern holds

One prompt could be a fluke — let's re-run with a different question to see if the same tone pattern shows up again.

In [6]:
second_prompt = "How do you mentally prepare a new hire for their first high-pressure project?"

results_2 = compare_roles(second_prompt)

for role_name, res in results_2.items():
    print("=" * 90)
    print(f"ROLE: {role_name}")
    print("=" * 90)
    print(res["text"])
    print()

ROLE: senior_data_analyst
To mentally prepare a new hire for their first high-pressure project, the following steps can be taken:

1. **Orientation and Onboarding**:
   - Provide a comprehensive orientation that includes company culture, values, and expectations.
   - Introduce the new hire to key team members and stakeholders.

2. **Skill Assessment**:
   - Evaluate the new hire’s current skill set and identify areas for development.
   - Provide targeted training or resources to address any skill gaps.

3. **Project Overview**:
   - Clearly outline the project objectives, scope, and deliverables.
   - Explain the project’s importance and potential impact on the organization.

4. **Mentorship and Support**:
   - Assign a mentor or experienced team member to provide guidance and support.
   - Encourage regular check-ins to discuss progress and address concerns.

5. **Stress Management Techniques**:
   - Introduce stress management strategies, such as time management, prioritization, an

### Tone and quality differences of persona

| Aspect | Senior Data Analyst persona | Creative Writer persona |
|---|---|---|
| **Tone** | Dry, precise, matter-of-fact | Expressive, narrative, uses metaphor |
| **Structure** | Tends toward lists/structured points, leads with the "what" | Tends toward flowing prose, leads with a hook or scene-setting |
| **Vocabulary** | Plain, quantitative where possible ("X% of employees...") | Colorful, figurative ("a slow leak that drains a team's momentum...") |
| **Best use case** | Reports, dashboards, decision-support writing | Blog posts, culture decks, employer-branding copy |
| **Risk if used for the wrong audience** | Can feel cold/dry for a persuasive/emotional appeal | Can feel imprecise or unserious for a data-driven audience |